# Fusion Pipeline Run

Run the fusion experiment pipeline and capture metrics.

Steps:
- Run the fusion experiment script.
- Review fusion metrics and scores.
- Verify the fusion model artifact.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'metrics': {},
    'scores_preview': None,
    'model': None,
}

run([PY, 'src/scripts/run_fusion_experiment.py'])


In [ ]:
# Load fusion metrics.
metrics_path = REPO_ROOT / 'experiments' / 'fusion' / 'metrics' / 'metrics.json'
if metrics_path.exists():
    data = json.loads(metrics_path.read_text(encoding='utf-8'))
    summary['metrics'] = data
    print(data)
else:
    print('Missing:', metrics_path)


In [ ]:
# Preview fusion scores and model artifact.
import pandas as pd

scores_path = REPO_ROOT / 'experiments' / 'fusion' / 'fusion_scores.csv'
if scores_path.exists():
    df = pd.read_csv(scores_path)
    summary['scores_preview'] = df.head(10).to_dict(orient='records')
    print(df.head(10))
else:
    print('Missing:', scores_path)

model_path = REPO_ROOT / 'models' / 'fusion' / 'fusion_meta_model.pkl'
if model_path.exists():
    summary['model'] = {
        'path': str(model_path.relative_to(REPO_ROOT)),
        'size_mb': round(model_path.stat().st_size / 1024**2, 2),
    }
    print('Fusion model:', summary['model'])
else:
    print('Missing:', model_path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'execution_fusion_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
